# Chapter 14 — Bar resonances in the stellar halo: chevrons, ridges and streams

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/14_bar_halo.ipynb)

**Goal.** Chapter 13 showed how the rotating bar shapes the velocities of disk stars near the Sun. The stellar halo
passes through the inner Galaxy too: a halo star on an eccentric orbit crosses the bar region every few hundred Myr.
Gaia revealed fine structure in the phase space of the halo — "chevrons" in the plane of radius and radial velocity,
ridges of constant energy, groups of stars moving inwards — that was first read as the debris of the last major merger,
Gaia-Sausage/Enceladus (Chapter 12), but which a series of papers by Dillamore et al. (2023, 2024, 2025) attribute to
resonances with the bar. We measure these features in Gaia DR3, build both explanations, and confront them with
the data.

**You will learn**
1. to look for weak structure in phase space: unsharp masking and **asymmetry maps**, and how to judge their significance;
2. how phase mixing of merger debris makes **chevrons** in the plane of Galactocentric (spherical) radius $r$ [kpc] and radial
   velocity $v_r$ [km/s, positive outwards], and what it predicts;
3. to model the halo in a **slowing bar** by backward integration (Chapter 13) and to identify features with resonances;
4. to use the data to estimate the **pattern speed** of the bar;
5. why the bar **disperses** substructure in the plane of angular momentum $L_z$ (per unit mass, about the Galaxy's rotation
   axis $z$; kpc km/s) and energy $E$ (per unit mass; (km/s)$^2$), but keeps it together in the **Jacobi integral** $E - \Omega_b L_z$,
   where $\Omega_b$ is the bar's pattern speed, the angular velocity of its rotation [km/s/kpc] (Chapter 13).
6. to grow a bar without changing the Galaxy's mass, and to measure how it **spins up** an initially non-rotating halo.

**Data.** The file `halo_chevrons` of the tutorial's Zenodo record (https://doi.org/10.5281/zenodo.23646482): Gaia DR3 stars with
line-of-sight velocities and the XP-based parallaxes of Zhang, Green & Rix (2023), with the cuts of Dillamore et al. (2024)
— reliable XP parameters, parallax better than 10%, distance below 15 kpc (22.45 million stars) — of which we keep the
1.67 million on orbits with $|L_z| < 1000$ kpc km/s, i.e. not on disk-like orbits.

**Units and conventions** as in Chapters 11 and 13: kpc, km/s, $M_\odot$ (solar masses); Galactocentric Cartesian coordinates
$(x, y, z)$ with $z$ towards the North Galactic Pole, $(v_x, v_y, v_z)$ the corresponding velocity components, and the Sun at $x = -R_0$, $y = 0$, where $R_0 = 8.122$ kpc; the Galaxy and the bar rotate
clockwise; we quote **prograde** angular momentum, $L_z > 0$ for motion with the disk, and positive $\Omega_b$.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object

import hashlib, urllib.request, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks

# Zenodo record of the tutorial data (as in Chapter 11)
ZENODO = 'https://zenodo.org/records/23646482/files/{}?download=1'
MD5 = {'solar_neighbourhood': 'f6119c98ae50ded14fab91d1d4c071fd', 'halo_gse': '1c0533ce89d6d95ff67406dcf2a95f31',
       'disc_giants': '12f6a9ca0ac3d8ecdcd71988267b8ea7', 'rotation_curves': 'a41c95e4ecc31b3ab3bde99880736c6e',
       'dsph_gaia': '0b135e7f08816209b379f892196307b2', 'halo_chevrons': '0c3dd8375d33f3a24bf8290f0b4c5098'}


def load_data(name, cache=Path('data')):
    """Download data/<name>.parquet from Zenodo once, check its MD5, and return it as a pandas DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} from Zenodo ...')
        urllib.request.urlretrieve(ZENODO.format(path.name), path)
    md5 = hashlib.md5(path.read_bytes()).hexdigest()
    assert md5 == MD5[name], f'{path} is corrupt (md5 {md5}); delete it and rerun'
    return pd.read_parquet(path)


R0, z_sun, v_sun = 8.122, 0.0208, (12.9, 245.6, 7.78)      # Chapter 11
K = 4.740470463533348
GYR = 0.9778                                                # Gyr per AGAMA time unit

## 14.1 Fine structure in the local halo

We convert the observables to Galactocentric coordinates as in Chapter 11, and compute the spherical radius $r$ and
radial velocity $v_r$, the cylindrical radius $R = \sqrt{x^2 + y^2}$, radial velocity $v_R$ (positive outwards) and rotation velocity
$v_\phi$ (positive in the direction of Galactic rotation), the prograde angular momentum $L_z = R\,v_\phi$, and the energy
$E = \frac12|\mathbf v|^2 + \Phi(\mathbf x)$ (with $\mathbf x$, $\mathbf v$ position and velocity and $\Phi$ the potential) in
the axisymmetrised Milky Way model of Chapter 13 (Hunter et al. 2024), which we rebuild here.

In [ ]:
from agama.py.example_mw_bar_potential import makeBarDensity

with np.errstate(all='ignore'):        # as in Chapter 13
    dens_bar = makeBarDensity()
    disks = [dict(type='Disk', surfaceDensity=1.332e9, scaleRadius=2.0, scaleHeight=0.3, innerCutoffRadius=2.7),
             dict(type='Disk', surfaceDensity=8.97e8, scaleRadius=2.8, scaleHeight=0.9, innerCutoffRadius=2.7),
             dict(type='Disk', surfaceDensity=5.81e7, scaleRadius=7, scaleHeight=-0.085, innerCutoffRadius=4),
             dict(type='Disk', surfaceDensity=2.68e9, scaleRadius=1.5, scaleHeight=-0.045, innerCutoffRadius=12)]
    spheroids = [dict(type='Spheroid', densitynorm=2.774e11, gamma=0, beta=0, alpha=1, outerCutoffRadius=8.682e-6,
                      cutoffStrength=0.1704),
                 dict(type='Plummer', mass=4.1e6, scaleRadius=1e-3),
                 dict(type='Spheroid', mass=6.1e7, gamma=0.71, beta=4, alpha=1, axisRatioZ=0.73, scaleRadius=0.0059,
                      outerCutoffRadius=0.1),
                 dict(type='Spheroid', densityNorm=2.00583e12, gamma=0, beta=0, alpha=1, axisRatioZ=0.37,
                      outerCutoffRadius=0.00506, cutoffStrength=0.72),
                 dict(type='Spheroid', densityNorm=1.53e12, gamma=0, beta=0, alpha=1, axisRatioZ=0.37,
                      outerCutoffRadius=0.0246, cutoffStrength=0.79)]
    pot_sph = agama.Potential(type='Multipole', density=agama.Density(*spheroids), lmax=12, gridSizeR=36, rmin=1e-4, rmax=1000)
    cyl = dict(type='CylSpline', density=agama.Density(dens_bar, *disks), gridSizeR=30, gridSizez=32,
               Rmin=0.1, Rmax=200, zmin=0.05, zmax=200)
    pot_bar = agama.Potential(pot_sph, agama.Potential(mmax=8, **cyl))     # bar along x, static
    pot_axi = agama.Potential(pot_sph, agama.Potential(mmax=0, **cyl))     # its m = 0 part

h = load_data('halo_chevrons')
col = lambda c: h[c].to_numpy(float)
l, b, pml, pmb = agama.transformCelestialCoords(agama.fromICRStoGalactic, np.radians(col('ra')), np.radians(col('dec')),
                                                col('pmra'), col('pmdec'))
x, y, z, vx, vy, vz = agama.getGalactocentricFromGalactic(l, b, 1 / col('parallax'), pml * K, pmb * K, col('radial_velocity'),
                                                          galcen_distance=R0, z_sun=z_sun, galcen_v_sun=v_sun)
r = np.sqrt(x**2 + y**2 + z**2)
R = np.hypot(x, y)
v_r = (x * vx + y * vy + z * vz) / r                 # spherical radial velocity
v_R = (x * vx + y * vy) / R                          # cylindrical radial velocity
Lz = -(x * vy - y * vx)                              # prograde angular momentum (> 0 for motion with the disk)
v_phi = Lz / R
E = pot_axi.potential(np.column_stack([x, y, z])) + 0.5 * (vx**2 + vy**2 + vz**2)
dist = np.sqrt((x + R0)**2 + y**2 + (z - z_sun)**2)  # distance from the Sun
print(f'{len(h):,} stars; heliocentric distance: median {np.median(dist):.1f} kpc, 90% below {np.percentile(dist, 90):.1f} kpc')

### Inward and outward

A halo in equilibrium has as many stars moving out as moving in, at every position. Its distribution function depends on the
velocity only through the integrals of motion — the energy $E$, the angular momentum $L_z$ and the actions $J_r$, $J_z$ (Chapter 9) —
and these do not change when the radial and vertical velocities both change sign, $(v_R, v_z) \to (-v_R, -v_z)$, at fixed $v_\phi$:
that is the same orbit run backwards in the meridional plane. Summed over $v_z$, the distribution of $v_R$ is therefore
symmetric. So the simplest test for departures from equilibrium is to compare the distribution of $v_R$ with its mirror image
$-v_R$. We take stars within 2.5 kpc of the Sun in
three bins of rotation velocity and count stars at $-v_R$ and at $+v_R$ in bins of $|v_R|$; the difference divided by its
Poisson uncertainty, $(N_- - N_+)/\sqrt{N_- + N_+}$, is the significance of the asymmetry, where $N_-$ and $N_+$ are the numbers of
stars with $v_R$ in the bin around $-|v_R|$ and around $+|v_R|$.

In [ ]:
local = dist < 2.5
bins = np.arange(-400, 401, 10)
c = 0.5 * (bins[1:] + bins[:-1])
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
signif = {}
for (lo, hi), colr in zip([(-50, 50), (50, 150), (-150, -50)], ['k', 'C1', 'C0']):
    m = local & (v_phi > lo) & (v_phi < hi)
    n, _ = np.histogram(v_R[m], bins)
    ax[0].plot(c, n / n.max(), color=colr, label=f'{lo} < $v_\\phi$ < {hi} km/s ({m.sum():,} stars)')
    ax[0].plot(-c, n / n.max(), color=colr, ls=':', lw=1)                 # mirror image
    n_in, n_out = n[c < 0][::-1], n[c > 0]                                # at -|v_R| and +|v_R|
    signif[(lo, hi)] = (n_in - n_out) / np.sqrt(n_in + n_out + 1)
    ax[1].plot(c[c > 0], signif[(lo, hi)], color=colr)
ax[0].set_xlabel('$v_R$ [km/s]'); ax[0].set_ylabel('normalised counts'); ax[0].legend(fontsize=8)
ax[1].axhline(0, color='0.5'); ax[1].set_xlabel('$|v_R|$ [km/s]'); ax[1].set_ylabel('$(N_- - N_+) / \\sqrt{N_- + N_+}$')
plt.tight_layout(); plt.show()
for k, s in signif.items():
    print(f'{k[0]:5d} < v_phi < {k[1]:4d}: largest inward excess {s.max():+.1f} sigma, largest outward excess {s.min():+.1f} sigma')
assert signif[(-50, 50)].max() > 5 and np.abs(signif[(-150, -50)]).max() < 4

**Figure 14.1.** Left: distributions of $v_R$ (solid) and their mirror images (dotted) for stars within 2.5 kpc of the Sun in three bins
of $v_\phi$. Right: significance of the excess of inward-moving ($v_R < 0$) over outward-moving stars as a function of $|v_R|$.

The halo near the Sun is not symmetric (Figure 14.1). Among non-rotating stars ($|v_\phi| < 50$ km/s) there are significantly
more stars moving **inwards** than outwards — 8 standard deviations at $|v_R| \approx 150$ km/s; slowly prograde stars
($50 < v_\phi < 150$ km/s) show a weaker inward excess at $|v_R| \approx 170$–200 km/s, and retrograde stars no significant asymmetry
at all. Dillamore et al. (2025) identified the inward-moving groups as two "dynamical streams" of the halo. Notice that this test needs no
model and no assumption about the selection function: the selection of Gaia is (very nearly) the same for a star moving
in and for its mirror image moving out.

### Chevrons and asymmetry maps

The same test works at every Galactocentric radius. For stars in bins of $L_z$ we map, in the plane of $r$ and $|v_r|$, the
asymmetry

$$ A(r, |v_r|) = \frac{N(r, -|v_r|) - N(r, +|v_r|)}{N(r, -|v_r|) + N(r, +|v_r|)} , $$

where $N(r, \mp|v_r|)$ is the number of stars in the cell at radius $r$ moving inwards ($v_r = -|v_r|$) or outwards ($v_r = +|v_r|$);
$A$ is positive (red) where inward-moving stars dominate. The halo near the Sun spans $r \approx 4$–14 kpc.

In [ ]:
r_edges = np.arange(3.875, 14.2, 0.25)          # bins centred on 4.0, 4.25, ..., 14.0 kpc
v_edges = np.linspace(0, 400, 31)               # |v_r| bins of 13.3 km/s
r_c, v_c = 0.5 * (r_edges[1:] + r_edges[:-1]), 0.5 * (v_edges[1:] + v_edges[:-1])
Lz_bins = {'retrograde, $-500 < L_z < 0$': (-500, 0), '$0 < L_z < 500$': (0, 500), '$500 < L_z < 1000$': (500, 1000)}


def asymmetry_data(lo, hi):
    """Smoothed asymmetry map A(r, |v_r|) and the number of stars per cell for lo < L_z < hi."""
    s = (Lz > lo) & (Lz < hi)
    n_in, _, _ = np.histogram2d(r[s], -v_r[s], bins=[r_edges, v_edges])
    n_out, _, _ = np.histogram2d(r[s], v_r[s], bins=[r_edges, v_edges])
    sm = lambda a: gaussian_filter(a, 0.7)                       # light smoothing, ~0.2 kpc x 10 km/s
    return (sm(n_in) - sm(n_out)) / np.maximum(sm(n_in + n_out), 1), n_in + n_out


A_data, N_data = {}, {}
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8), sharey=True)
for a, (lab, (lo, hi)) in zip(ax, Lz_bins.items()):
    A_data[lab], N_data[lab] = asymmetry_data(lo, hi)
    a.imshow(np.where(N_data[lab] > 30, A_data[lab], np.nan).T, origin='lower', aspect='auto', cmap='RdBu_r',
             vmin=-0.3, vmax=0.3, extent=[r_edges[0], r_edges[-1], 0, 400])
    a.axvline(R0, color='k', ls=':', lw=1)
    a.set_title(f'Gaia DR3: {lab}'); a.set_xlabel('$r$ [kpc]')
ax[0].set_ylabel('$|v_r|$ [km/s]')
plt.tight_layout(); plt.show()
for lab in Lz_bins:
    w = N_data[lab] > 30
    print(f'{lab:28s}: mean asymmetry {A_data[lab][w].mean():+.3f} over {w.sum()} cells')

**Figure 14.2.** Asymmetry between inward- and outward-moving stars, $A(r, |v_r|)$, in three bins of prograde angular momentum
(red: more stars moving inwards); cells with fewer than 30 stars are blank. The dotted line marks $R_0$.

For $0 < L_z < 500$ kpc km/s a diagonal band of inward-moving stars runs from $(r, |v_r|) \approx (6.5, 250)$ to $(10, 80)$
(Figure 14.2, middle): this is one arm of a **chevron**, a feature that in the $(r, v_r)$ plane looks like a "<" whose two arms are
the inward- and outward-moving halves of a family of orbits with similar energies — and here one arm is much stronger
than the other. At $500 < L_z < 1000$ the asymmetry has the opposite sign (blue), and among retrograde stars it is weak.

### A ridge in energy

The third feature is in the plane of $L_z$ and $E$. The density there is smooth on large scales; we subtract a smoothed
version of its logarithm ("unsharp masking") to bring out structure on scales below $\approx 0.03 \times 10^5$ (km/s)$^2$ in energy.

In [ ]:
H, Le, Ee = np.histogram2d(Lz, E / 1e5, bins=[200, 200], range=[[-1000, 1000], [-2.0, -1.0]])
U = gaussian_filter(np.log((H + 1) / (gaussian_filter(H, 5) + 1)), 1)       # log density minus its smoothed version
E_c, L_c = 0.5 * (Ee[1:] + Ee[:-1]), 0.5 * (Le[1:] + Le[:-1])
prof_pro = U[(L_c > 0) & (L_c < 500)].mean(axis=0)            # residual averaged over 0 < L_z < 500 (prograde)
prof_ret = U[(L_c > -500) & (L_c < 0)].mean(axis=0)           # and over -500 < L_z < 0 (retrograde)
win = (E_c > -1.55) & (E_c < -1.15)                           # above the lower edge of the distribution
pk, props = find_peaks(prof_pro[win], prominence=0.005)       # local maxima and how far they stand out
E_ridge = E_c[win][pk[np.argmax(props['prominences'])]]       # the most prominent one

fig, ax = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw=dict(width_ratios=[1, 1, 0.5]))
ax[0].imshow(np.log10(H.T + 1), origin='lower', aspect='auto', extent=[-1000, 1000, -2, -1], cmap='Greys')
ax[1].imshow(U.T, origin='lower', aspect='auto', extent=[-1000, 1000, -2, -1], cmap='Greys', vmin=-0.15, vmax=0.15)
ax[2].plot(prof_pro, E_c, 'k', label='$0 < L_z < 500$'); ax[2].plot(prof_ret, E_c, color='0.6', label='$-500 < L_z < 0$')
ax[2].axhline(E_ridge, color='C3', ls='--'); ax[2].set_ylim(-2, -1); ax[2].set_xlim(-0.3, 0.2); ax[2].legend(fontsize=8)
for a in ax[:2]:
    a.set_xlabel('$L_z$ [kpc km/s]')
ax[0].set_ylabel('$E$ [$10^5$ (km/s)$^2$]'); ax[2].set_xlabel('mean residual')
plt.tight_layout(); plt.show()
print(f'ridge at E = {E_ridge:.3f} x 1e5 (km/s)^2')
assert -1.5 < E_ridge < -1.35

**Figure 14.3.** Left: density of the sample in the $(L_z, E)$ plane (log scale). Middle: the same after unsharp masking (dark:
overdense). Right: the residual averaged over $0 < L_z < 500$ (black) and $-500 < L_z < 0$ kpc km/s (grey) as a function of energy; the
dashed line marks the most prominent prograde peak.

A narrow ridge at constant energy, $E \approx -1.43 \times 10^5$ (km/s)$^2$, runs across the prograde side (Figure 14.3); the retrograde
residual is flatter at that energy. This is the
feature that Dillamore et al. (2023) found in the Gaia DR3 radial-velocity sample. A ridge at *constant energy* is hard to
make with debris of a merger, whose stars spread along a range of energies; we will see that the bar makes it naturally.

### Asymmetry in the $(L_z, E)$ plane

The inward–outward test of Figure 14.1 can be applied in the $(L_z, E)$ plane as well. We colour each cell by

$$ A(L_z, E) = \frac{N_{\rm in}(L_z, E) - N_{\rm out}(L_z, E)}{N_{\rm in}(L_z, E) + N_{\rm out}(L_z, E)} , $$

where $N_{\rm in}$ and $N_{\rm out}$ are the numbers of stars in the cell moving inwards ($v_r < 0$) and outwards ($v_r > 0$), with $v_r$
the spherical Galactocentric radial velocity. For a halo in equilibrium $A = 0$ in every cell, by the argument of Figure 14.1:
reversing $v_R$ and $v_z$ reverses $v_r$ and changes neither $E$ nor $L_z$. Figure 14.3 shows where stars pile up in $(L_z, E)$;
this map shows which of them are out of equilibrium.

In [ ]:
kw = dict(bins=[100, 100], range=[[-1000, 1000], [-2.0, -1.0]])        # cells of 20 kpc km/s x 0.01e5 (km/s)^2
n_in, Le2, Ee2 = np.histogram2d(Lz[v_r < 0], E[v_r < 0] / 1e5, **kw)
n_out, _, _ = np.histogram2d(Lz[v_r > 0], E[v_r > 0] / 1e5, **kw)
sm = lambda a: gaussian_filter(a, 1.5)                                  # smooth over ~30 kpc km/s x 0.015e5 (km/s)^2
A_LE = (sm(n_in) - sm(n_out)) / np.maximum(sm(n_in + n_out), 1)
L_c2, E_c2 = 0.5 * (Le2[1:] + Le2[:-1]), 0.5 * (Ee2[1:] + Ee2[:-1])

fig, ax = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw=dict(width_ratios=[1, 0.6]))
im = ax[0].imshow(np.where(sm(n_in + n_out) > 20, A_LE, np.nan).T, origin='lower', aspect='auto',   # blank: < 20 stars per cell
                  extent=[-1000, 1000, -2, -1], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
ax[0].contour(L_c2, E_c2, np.log10(sm(n_in + n_out).T + 1), levels=[1.5, 2.5, 3.5], colors='0.4', linewidths=0.6)
ax[0].axhline(E_ridge, color='k', ls=':', lw=1)
plt.colorbar(im, ax=ax[0], label='$A(L_z, E)$')
ax[0].set_xlabel('$L_z$ [kpc km/s]'); ax[0].set_ylabel('$E$ [$10^5$ (km/s)$^2$]')
prof_A = {}
for (lo, hi), colr in zip([(0, 500), (-500, 0), (500, 1000)], ['k', '0.6', 'C1']):
    s = (L_c2 > lo) & (L_c2 < hi)
    ni, no = gaussian_filter(n_in[s].sum(0), 1), gaussian_filter(n_out[s].sum(0), 1)   # summed over the L_z bin
    prof_A[(lo, hi)] = (ni - no) / np.maximum(ni + no, 1)
    ax[1].plot(np.where(ni + no > 50, prof_A[(lo, hi)], np.nan), E_c2, color=colr, label=f'{lo} < $L_z$ < {hi}')
ax[1].axvline(0, color='0.5'); ax[1].axhline(E_ridge, color='C3', ls='--')
ax[1].set_ylim(-2, -1); ax[1].set_xlim(-0.35, 0.35); ax[1].legend(fontsize=8)
ax[1].set_xlabel('$A(E)$'); ax[1].set_ylabel('$E$ [$10^5$ (km/s)$^2$]')
plt.tight_layout(); plt.show()

Phi_sun = pot_axi.potential([-R0, 0, z_sun]) / 1e5
for k, a in prof_A.items():
    i, j = np.argmax(a), np.argmin(a)
    print(f'{k[0]:5d} < Lz < {k[1]:4d}: max A = {a[i]:+.2f} at E = {E_c2[i]:.2f}, min A = {a[j]:+.2f} at E = {E_c2[j]:.2f}')
E_in = E_c2[np.argmax(prof_A[(0, 500)])]
print(f'Phi at the Sun = {Phi_sun:.3f} x 1e5 (km/s)^2: a star at the Sun with E = {E_in:.2f} has speed '
      f'{np.sqrt(2e5 * (E_in - Phi_sun)):.0f} km/s')
assert prof_A[(0, 500)].max() > 0.1 and E_in > E_ridge

**Figure 14.4.** Left: asymmetry $A(L_z, E)$ between inward- ($v_r < 0$) and outward-moving ($v_r > 0$) stars in the $(L_z, E)$ plane
(red: more stars moving inwards); cells with fewer than 20 stars after smoothing are blank, grey contours show the number of stars per
cell ($10^{1.5}$, $10^{2.5}$, $10^{3.5}$), and the dotted line marks the energy of the ridge of Figure 14.3. Right: $A$ as a function of $E$
for three bins of $L_z$ (in kpc km/s); the dashed line marks the ridge energy.

The departure from equilibrium is confined to a narrow range of energy (Figure 14.4). For $0 < L_z < 500$ kpc km/s inward-moving
stars outnumber outward-moving ones by $A \approx 0.21$ at $E \approx -1.29 \times 10^5$ (km/s)$^2$, a band just above the ridge of
Figure 14.3 ($E \approx -1.43 \times 10^5$), while at the ridge energy itself $A \approx 0$. The potential at the Sun is
$\Phi \approx -1.45 \times 10^5$ (km/s)$^2$, so a star near the Sun with $E = -1.29 \times 10^5$ (km/s)$^2$ moves at
$\sqrt{2(E - \Phi)} \approx 181$ km/s: these are the inward-moving, non-rotating stars of Figure 14.1. At almost the same energy the
next bin, $500 < L_z < 1000$, has the opposite sign ($A \approx -0.18$ at $E \approx -1.27 \times 10^5$), the sign change between the
middle and right panels of Figure 14.2; retrograde stars stay within $|A| \le 0.05$ at all energies. An asymmetry that is localised
in energy at fixed $L_z$, and changes sign with $L_z$, is what a resonance produces; §14.3 builds the model.

## 14.2 First explanation: phase mixing of merger debris

Belokurov et al. (2023) found the chevrons in the radial phase space of Gaia stars and interpreted them as the debris of
Gaia-Sausage/Enceladus (Chapter 12). Stars released by a satellite on a radial orbit have slightly different energies,
hence different radial periods; as they phase-mix (Chapter 6), the debris wraps into a series of nested chevrons in the
$(r, v_r)$ plane, more of them as time goes on. A minimal model: 20 000 stars released together at $r \approx 25$ kpc on nearly
radial orbits, integrated in the axisymmetric potential.

In [ ]:
rng = np.random.default_rng(2)
N = 20_000
ic = np.column_stack([np.array([-25., 0, 5.]) + rng.normal(0, 1.5, (N, 3)),        # positions [kpc]
                      np.array([20., 15., -20.]) + rng.normal(0, 15, (N, 3))])      # velocities [km/s]
debris = agama.orbit(potential=pot_axi, ic=ic, time=6.0, trajsize=4, verbose=False)  # snapshots at t = 0, 2, 4, 6
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), sharey=True)
asym_debris = []
for a, k in zip(ax, (1, 2, 3)):
    w = np.array([traj[k] for traj in debris[:, 1]], dtype=float)
    rr = np.linalg.norm(w[:, :3], axis=1); vrr = np.sum(w[:, :3] * w[:, 3:], axis=1) / rr
    a.hist2d(rr, vrr, bins=[150, 150], range=[[0, 35], [-450, 450]], cmap='Greys', norm=LogNorm())
    a.set_title(f'debris after {2 * k * GYR:.1f} Gyr'); a.set_xlabel('$r$ [kpc]')
    s = (rr > 5) & (rr < 12)
    asym_debris.append((np.sum(vrr[s] < 0) - np.sum(vrr[s] > 0)) / s.sum())
ax[0].set_ylabel('$v_r$ [km/s]')
plt.tight_layout(); plt.show()
print('debris at 5 < r < 12 kpc: (N_in - N_out) / N =', np.round(asym_debris, 3))
assert abs(asym_debris[-1]) < 0.05

**Figure 14.5.** Radial phase space of the debris of a disrupted satellite in the axisymmetric potential, 2, 3.9 and 5.9 Gyr after
release (log counts).

Phase mixing indeed makes chevrons (Figure 14.5): nested "<" shapes, tighter and more numerous with time, each made of stars of
nearly the same energy. But once the debris is a few radial periods old, each chevron has as many stars moving in as
out — the asymmetry at 5–12 kpc is below 2% after 4 Gyr. Old merger debris predicts **symmetric** chevrons, a spread in
energy rather than a single ridge, and no particular preference for prograde orbits. The Gaia halo shows the opposite on
all three counts (§14.1). Dillamore et al. (2024) used exactly these arguments.

## 14.3 Second explanation: resonances with a slowing bar

Halo stars that pass through the inner Galaxy feel the bar. Orbits in resonance with it, $m(\Omega_\phi - \Omega_b) = l\,\Omega_R$
(Chapter 13; $m$ is the azimuthal wavenumber of the bar, 2 for its main part, $l$ an integer, $\Omega_b$ the pattern speed, and
$\Omega_\phi$, $\Omega_R$ the azimuthal and radial frequencies of the orbit, $\Omega_\phi$ positive for prograde motion), are pushed coherently;
others are not. Corotation ($l = 0$), the outer Lindblad resonance (OLR; $l = -1$, $m = 2$) and the 1:1 resonance ($\Omega_\phi - \Omega_b = -\Omega_R$) exist for eccentric
halo orbits too, at energies set by $\Omega_b$. A bar that **slows down** — as dynamical friction against the dark halo makes it do —
sweeps its resonances outwards through the halo, captures stars and drags them along (Chiba, Friske & Schönrich 2021).

We build the model of Chapter 13 with two changes: the pattern speed decreases with time as $\dot\Omega_b = -\eta\,\Omega_b^2$, the law of
Chiba et al. (2021), so that

$$ \Omega_b(t) = \frac{\Omega_{b,i}}{1 + \eta\,\Omega_{b,i}\,(t - t_i)}, \qquad \phi_b(t) = \phi_{b,i} - \frac{1}{\eta}\ln\left[1 + \eta\,\Omega_{b,i}(t - t_i)\right] $$

where $t$ is time ($t = 0$ today, negative in the past), $\dot\Omega_b = d\Omega_b/dt$, $\eta$ a dimensionless constant that sets the rate of slowing
($\Omega_b\,t$ is an angle, so $\eta\,\Omega_b\,t$ is dimensionless), $t_i$ the time when the slowing starts, $\Omega_{b,i}$ and $\phi_{b,i}$ the pattern speed and
the bar angle at $t_i$, and $\phi_b(t)$ the angle of the bar's major axis measured anticlockwise from the $x$ axis
(minus because the bar turns clockwise); and the bar grows longer as it slows, its length scaling as $1/\Omega_b$, so that the
ratio of corotation radius to bar length stays fixed (Dillamore et al. 2024 use the same assumption). The `scale` modifier
stretches the non-axisymmetric part by the length factor $S(t) = \Omega_{b,\rm now}/\Omega_b(t)$, with $\Omega_{b,\rm now}$ today's pattern speed, and the `rotation` modifier turns it by $\phi_b(t)$. We choose
$\eta = 0.0036$, which gives today's slowing rate of Chiba et al. (2021), $\dot\Omega_b = -4.5$ km/s/kpc/Gyr for $\Omega_b = 35$ km/s/kpc,
and run for 4 time units (3.9 Gyr). The bar then started at $\Omega_b \approx 71$ km/s/kpc (printed below), i.e. half as long as today.

In [ ]:
bar_angle = np.radians(-28.0)          # bar orientation today (Chapter 13)
T, eta = 4.0, 0.0036


def slowing_bar(Om_now, T=T, eta=eta):
    """Barred potential whose pattern speed falls to Om_now today (t = 0) after a time T; the bar grows over the first time unit."""
    tt = np.linspace(-T - 0.5, 0.5, 4001)
    Om_i = Om_now / (1 - eta * Om_now * T)                                # pattern speed at t = -T
    Om_t = Om_i / (1 + eta * Om_i * (tt + T))
    angle = bar_angle - (np.log(1 + eta * Om_i * (tt + T)) - np.log(1 + eta * Om_i * T)) / eta   # = bar_angle at t = 0
    S = Om_now / Om_t                                                     # bar length ~ 1 / Omega_b
    xi = np.clip(2 * (tt + T) - 1, -1, 1)
    A = 3 / 16 * xi**5 - 5 / 8 * xi**3 + 15 / 16 * xi + 0.5               # Dehnen (2000) growth over one time unit
    scale = np.column_stack([tt, A * S, S])                               # [t, amplitude, length factor]
    scale_neg = scale * [1, -1, 1]
    rot = np.column_stack([tt, angle])
    return agama.Potential(pot_axi, agama.Potential(potential=pot_bar, scale=scale, rotation=rot),
                           agama.Potential(potential=pot_axi, scale=scale_neg, rotation=rot)), Om_i


pot35, Om_i = slowing_bar(35.0)
print(f'pattern speed 3.9 Gyr ago: {Om_i:.1f} km/s/kpc; slowing rate today: {-eta * 35**2 / GYR:.1f} km/s/kpc/Gyr')
p = [3.0, 1.0, 0.5]
assert abs(pot35.potential(p, t=-T - 0.2) / pot_axi.potential(p) - 1) < 1e-10        # no bar before it grows
assert abs(pot35.potential(np.array([3.0 * np.cos(bar_angle), 3.0 * np.sin(bar_angle), 0.5]), t=0)
           / pot_bar.potential([3.0, 0, 0.5]) - 1) < 1e-8                          # the bar today, at its angle

The halo before the bar is described by a smooth, radially anisotropic distribution function (DF) $f_0$ of the actions,
AGAMA's `DoublePowerLaw` (Chapter 10), in the axisymmetric potential; it is an exact equilibrium there. Its argument
$\mathbf J = (J_r, J_z, J_\phi)$ are the radial, vertical and azimuthal actions, and $h(\mathbf J)$, $g(\mathbf J)$ are weighted sums of them
that measure the size of an orbit near the centre and far out. Recall its form,

$$ f_0(\mathbf J) \propto \left[1 + \frac{J_0}{h(\mathbf J)}\right]^{\Gamma} \left[1 + \frac{g(\mathbf J)}{J_0}\right]^{-B}, \qquad
   h = h_r J_r + h_z J_z + (3 - h_r - h_z)|J_\phi|, \quad g = g_r J_r + g_z J_z + (3 - g_r - g_z)|J_\phi| , $$

(with AGAMA's default steepness of the transition, 1), where the actions are in kpc km/s,
$J_0 = 500$ kpc km/s the break action (`J0`), $\Gamma = 0$ and $B = 3.5$ the inner and outer slopes (`slopeIn`, `slopeOut`), and
$h_r = g_r = 0.6$, $h_z = g_z = 1.2$ the weights of the actions (`coefJrIn`, `coefJrOut`, `coefJzIn`, `coefJzOut`). A weight below 1 on $J_r$
makes $f_0$ fall more slowly with $J_r$ than with the other actions, so the halo is radially anisotropic. The normalisation does not
matter: we only use the ratio of the DF today, $f$, to $f_0$. As in Chapter 13, we
predict today's $f$ by **backward integration**: for each point of a grid, integrate the orbit back through the bar's history
to $t = -T$ ($T$ = 4 time units) and evaluate $f_0$ at the starting point; $f_0(\mathbf J)$ is the DF before the bar, a function of the actions
$\mathbf J = (J_r, J_z, J_\phi)$ in the axisymmetric potential, and $f$ the DF today. The ratio $f/f_0$ shows what the bar did. The grid lies along the
Sun–Galactic centre line in the plane, with $4 < r < 14$ kpc and $|v_r| < 400$ km/s; to represent halo stars we add vertical
velocities $v_\theta = -100, 0, 100$ km/s ($v_\theta$ is the velocity along the polar angle $\theta$, measured from the North Galactic Pole;
in the plane $v_\theta = -v_z$) and two values of $L_z$ inside each $L_z$ bin, and sum $f$ over them.

In [ ]:
df_halo = agama.DistributionFunction(type='DoublePowerLaw', norm=1, J0=500, slopeIn=0, slopeOut=3.5,
                                     coefJrIn=0.6, coefJzIn=1.2, coefJrOut=0.6, coefJzOut=1.2)   # coefJr < 1: radial orbits
af = agama.ActionFinder(pot_axi)
f0 = lambda xv: df_halo(af(xv))

r_g = r_c                                        # same radii as the data maps
v_g = np.concatenate([-v_c[::-1], v_c])          # signed v_r: inward then outward
vth_g = np.array([-100., 0., 100.])


def model_grid(Lz_values):
    """Initial conditions on the grid (r, v_r, v_theta, L_z) along the Sun-Galactic centre line in the plane."""
    Rg, Vg, Tg, Lg = np.meshgrid(r_g, v_g, vth_g, Lz_values, indexing='ij')
    n = Rg.size
    pos = np.column_stack([-Rg.ravel(), np.zeros(n), np.zeros(n)])            # at azimuth 180 deg (towards the Sun)
    vel = np.column_stack([-Vg.ravel(),                                        # v_r > 0 is along -x here
                           Lg.ravel() / Rg.ravel(),                            # prograde (clockwise) is +y at this azimuth
                           -Tg.ravel()])                                       # v_theta = -v_z in the plane
    return np.column_stack([pos, vel]), Rg.shape


def f_today(pot, Lz_values):
    """f(r, v_r) today (summed over v_theta and L_z) by backward integration, and the no-bar f0 on the same grid."""
    ic, shape = model_grid(Lz_values)
    out = agama.orbit(potential=pot, ic=ic, time=-T, timestart=0, trajsize=1, verbose=False)
    start = np.array([traj[-1] for traj in out[:, 1]], dtype=float)
    return f0(start).reshape(shape).sum(axis=(2, 3)), f0(ic).reshape(shape).sum(axis=(2, 3))


def asymmetry_model(f):
    """A(r, |v_r|) from f on the signed v_r grid: inward half is the first half, mirrored."""
    n = len(v_c)
    f_in, f_out = f[:, :n][:, ::-1], f[:, n:]
    return gaussian_filter((f_in - f_out) / (f_in + f_out), 0.7)


t0 = time.time()
f35, fa35 = f_today(pot35, [150., 350.])
print(f'{len(r_g) * len(v_g) * len(vth_g) * 2:,} orbits in {time.time() - t0:.0f} s')

# resonances today for the points of the grid: (Omega_phi - Omega_b) / Omega_r in the axisymmetric potential
ic_res, shape_res = model_grid([250.])
_, Om = af(ic_res, frequencies=True)
ratio_res = ((np.abs(Om[:, 2]) - 35.0) / Om[:, 0]).reshape(shape_res)[:, :, 1, 0]      # at v_theta = 0

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].imshow(gaussian_filter(f35 / fa35, 1).T, origin='lower', aspect='auto', extent=[r_edges[0], r_edges[-1], -400, 400],
             cmap='RdBu_r', vmin=0.4, vmax=1.6)
cs = ax[0].contour(r_g, v_g, ratio_res.T, levels=[-1, -0.5, 0], colors=['k', 'C0', 'C3'], linewidths=1.2)
ax[0].clabel(cs, fmt={-1: '1:1', -0.5: 'OLR', 0: 'CR'}, fontsize=9)
ax[0].set_xlabel('$r$ [kpc]'); ax[0].set_ylabel('$v_r$ [km/s]'); ax[0].set_title('$f / f_0$, $0 < L_z < 500$, $\\Omega_b$ = 35 km/s/kpc today')
A35 = asymmetry_model(f35)
ax[1].imshow(A35.T, origin='lower', aspect='auto', extent=[r_edges[0], r_edges[-1], 0, 400], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
ax[1].set_xlabel('$r$ [kpc]'); ax[1].set_ylabel('$|v_r|$ [km/s]'); ax[1].set_title('model asymmetry $A(r, |v_r|)$')
plt.tight_layout(); plt.show()
print(f'model: f/f0 ranges from {np.percentile(f35 / fa35, 1):.2f} to {np.percentile(f35 / fa35, 99):.2f} (1st-99th percentile)')
assert np.percentile(f35 / fa35, 99) > 1.2 and np.percentile(f35 / fa35, 1) < 0.8

**Figure 14.6.** Backward-integration model of the halo in a slowing bar, for $0 < L_z < 500$ kpc km/s along the Sun–Galactic centre
line. Left: $f/f_0$ in the $(r, v_r)$ plane, smoothed by one cell (red: more stars than without the bar), with the resonance lines for today's pattern
speed, where $(\Omega_\phi - \Omega_b)/\Omega_R$ equals 0 (corotation, red), $-1/2$ (OLR, blue) and $-1$ (the 1:1 resonance, black). Right: the asymmetry between inward and outward motion.

The bar rearranges the halo into chevrons (Figure 14.6, left). They follow the resonance lines, and — unlike the phase-mixed
debris — they are strongly asymmetric: the inward arm of the corotation chevron is overdense, the outward arm underdense
(right). The reason is geometric (Dillamore et al. 2024): orbits trapped in a resonance keep a fixed orientation relative to
the bar, so at a given place relative to the bar — near the Sun, $28°$ behind the bar's major axis — they are always caught
on the same phase of their radial oscillation.

### Measuring the pattern speed

The resonances, and the chevrons with them, move outwards when the bar is slower. We run the same model for three present-day
pattern speeds, compare its asymmetry map with the data, and measure (i) the correlation coefficient of the two maps over the
cells with at least 50 stars, and (ii) the radius of the strongest inward band, where $A$ averaged over $100 < |v_r| < 200$ km/s
peaks.

In [ ]:
lab = '$0 < L_z < 500$'
Ad, good = A_data[lab], N_data[lab] > 50


def band_radius(A):
    """r of the maximum of A averaged over 100 < |v_r| < 200 km/s, for 5 < r < 12 kpc."""
    prof = A[:, (v_c > 100) & (v_c < 200)].mean(axis=1)
    w = (r_c > 5) & (r_c < 12)
    return r_c[w][np.argmax(prof[w])]


models = {35.0: A35}
for Om_now in (38.0, 32.0):
    t0 = time.time()
    models[Om_now] = asymmetry_model(f_today(slowing_bar(Om_now)[0], [150., 350.])[0])
    print(f'Omega_b = {Om_now}: {time.time() - t0:.0f} s')
corr = {Om: np.corrcoef(Ad[good], Am[good])[0, 1] for Om, Am in models.items()}
r_band = {Om: band_radius(Am) for Om, Am in models.items()}
print(f'data: inward band at r = {band_radius(Ad):.2f} kpc')
for Om in sorted(models):
    print(f'model Omega_b = {Om}: band at r = {r_band[Om]:.2f} kpc, correlation with the data {corr[Om]:+.2f}')

fig, ax = plt.subplots(1, 4, figsize=(22, 4.6), sharey=True)
ax[0].imshow(np.where(N_data[lab] > 30, Ad, np.nan).T, origin='lower', aspect='auto', extent=[r_edges[0], r_edges[-1], 0, 400],
             cmap='RdBu_r', vmin=-0.3, vmax=0.3)
ax[0].set_title('Gaia DR3, $0 < L_z < 500$')
for a, Om in zip(ax[1:], (38.0, 35.0, 32.0)):
    a.imshow(models[Om].T, origin='lower', aspect='auto', extent=[r_edges[0], r_edges[-1], 0, 400], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    a.set_title(f'model, $\\Omega_b$ = {Om:.0f}: corr = {corr[Om]:+.2f}')
for a in ax:
    a.axvline(band_radius(Ad), color='k', ls=':', lw=1); a.set_xlabel('$r$ [kpc]')
ax[0].set_ylabel('$|v_r|$ [km/s]')
plt.tight_layout(); plt.show()
assert corr[32.0] > corr[38.0] + 0.3 and r_band[38.0] < band_radius(Ad) < r_band[32.0] + 0.3

**Figure 14.7.** The asymmetry map of the data for $0 < L_z < 500$ kpc km/s (left) and of the slowing-bar model with three present-day
pattern speeds; the dotted line marks the radius of the strongest inward band in the data. The titles give the correlation
coefficient between model and data maps.

A bar of $\Omega_b = 38$ km/s/kpc puts the band of inward-moving stars 1.5 kpc too far in and does not correlate with the data;
$\Omega_b = 35$ is close, and $\Omega_b = 32$ matches both the position of the band and the overall pattern best (Figure 14.7). From
the chevrons alone we would estimate $\Omega_b \approx 32$–34 km/s/kpc, in agreement with Dillamore et al. (2025), who fitted resonant orbits to the
tracks of the halo streams and found $\Omega_b = 31.9^{+1.8}_{-1.9}$ km/s/kpc, and with the measurement from the bar itself by Clarke & Gerhard
(2022), $33.3 \pm 1.8$. This is a rough estimate: the model halo DF, the slowing history, the bar's amplitude and the potential
all shift the bands, which is why Dillamore et al. (2025) fitted the potential and $\Omega_b$ together.

The constant-speed bar of Chapter 13 makes weaker and differently placed features (Exercise 2); in the test-particle simulations
of Dillamore et al. (2024) also, the chevrons become prominent only when the bar slows.

### The asymmetry in $(L_z, E)$

The grid along the Sun–Galactic centre line cannot be compared with Figure 14.4, whose stars fill a volume several kpc across. Instead
we use the stars themselves: for each of 8000 random Gaia stars in the window $-500 < L_z < 1000$ kpc km/s, $-1.6 < E/10^5 < -1.0$
(km/s)$^2$ we compute today's $f$ by backward integration at the star's own position and velocity, and at its **mirror image**, the
same position with $(v_R, v_z) \to (-v_R, -v_z)$, which has the same $E$ and $L_z$ and the opposite $v_r$. With $f_{\rm in}$ and $f_{\rm out}$
the values of $f$ at whichever of the two points moves inwards and outwards, a star at that position and speed is expected to be
moving inwards with probability $f_{\rm in}/(f_{\rm in} + f_{\rm out})$, so

$$ a_\star = \frac{f_{\rm in} - f_{\rm out}}{f_{\rm in} + f_{\rm out}} $$

is its expected contribution to $(N_{\rm in} - N_{\rm out})/N$, and the mean of $a_\star$ over the stars in an energy bin is the model's
prediction for $A(E)$ of Figure 14.4. The stars sample the volume, the speeds and the selection function of the data exactly; the
bar alone decides the sign. Without the bar $a_\star = 0$ exactly, because the star and its mirror image have the same actions. We use the
best-matching pattern speed of Figure 14.7, $\Omega_b = 32$ km/s/kpc today.

In [ ]:
pot32 = slowing_bar(32.0)[0]
rng_s = np.random.default_rng(1)                    # own generator: leaves the clump of §14.4 unchanged
window = (E / 1e5 > -1.6) & (E / 1e5 < -1.0) & (Lz > -500) & (Lz < 1000)
pick = rng_s.choice(np.flatnonzero(window), 8000, replace=False)     # random Gaia stars in the window of Figure 14.4
xv = np.column_stack([x, y, z, vx, vy, vz])[pick]
uR = xv[:, :2] / R[pick, None]                                       # unit vector along R
xv_mirror = xv.copy()                                                # (v_R, v_z) -> (-v_R, -v_z): same E and L_z, v_r -> -v_r
xv_mirror[:, 3:5] -= 2 * v_R[pick, None] * uR
xv_mirror[:, 5] *= -1

t0 = time.time()
out = agama.orbit(potential=pot32, ic=np.vstack([xv, xv_mirror]), time=-T, timestart=0, trajsize=1,
                  accuracy=1e-5, verbose=False)                      # f changes by < 0.3% from the default 1e-8, 3x faster
start = np.array([traj[-1] for traj in out[:, 1]], dtype=float)
f_star, f_mirror = np.split(f0(start), 2)                            # f today at the star and at its mirror image
print(f'{2 * len(pick):,} orbits in {time.time() - t0:.0f} s')
inward = v_r[pick] < 0
f_in, f_out = np.where(inward, f_star, f_mirror), np.where(inward, f_mirror, f_star)
a_star = (f_in - f_out) / (f_in + f_out)          # expected (N_in - N_out) / N for stars at this position and speed
Lz_s, E_s = Lz[pick], E[pick] / 1e5

E_edges = np.arange(-1.6, -0.999, 0.025)
E_mid = 0.5 * (E_edges[1:] + E_edges[:-1])
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8), sharey=True)
A_mod_prof = {}
for a, (lo, hi), colr in zip(ax, [(-500, 0), (0, 500), (500, 1000)], ['0.6', 'k', 'C1']):
    s = (Lz_s > lo) & (Lz_s < hi)
    num, _ = np.histogram(E_s[s], E_edges, weights=a_star[s])
    cnt, _ = np.histogram(E_s[s], E_edges)
    A_mod_prof[(lo, hi)] = num / np.maximum(cnt, 1)
    err = a_star[s].std() / np.sqrt(np.maximum(cnt, 1))                 # sampling error of the mean
    a.fill_betweenx(E_mid, A_mod_prof[(lo, hi)] - err, A_mod_prof[(lo, hi)] + err, color=colr, alpha=0.3, lw=0)
    a.plot(A_mod_prof[(lo, hi)], E_mid, color=colr, label='model, $\\Omega_b$ = 32')
    a.plot(prof_A[(lo, hi)], E_c2, color=colr, ls=':', lw=2, label='Gaia DR3 (Figure 14.4)')
    a.axvline(0, color='0.5'); a.axhline(E_ridge, color='C3', ls='--', lw=1)
    a.set_xlim(-0.4, 0.4); a.set_ylim(-1.6, -1.0); a.set_xlabel('$A(E)$'); a.set_title(f'{lo} < $L_z$ < {hi} kpc km/s')
ax[0].set_ylabel('$E$ [$10^5$ (km/s)$^2$]'); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

for k, A in A_mod_prof.items():
    i, j = np.argmax(A), np.argmin(A)
    print(f'model {k[0]:5d} < Lz < {k[1]:4d}: max A = {A[i]:+.2f} at E = {E_mid[i]:.2f}, min A = {A[j]:+.2f} at E = {E_mid[j]:.2f}')
d = {k: np.interp(E_mid, E_c2, prof_A[k]) for k in A_mod_prof}       # data profile on the model's E bins
for k in A_mod_prof:
    print(f'{k}: correlation of model and data profiles {np.corrcoef(A_mod_prof[k], d[k])[0, 1]:+.2f}')

_, Om_s = af(xv, frequencies=True)                                  # (Omega_r, Omega_z, Omega_phi) in the axisymmetric potential
ratio_s = (np.abs(Om_s[:, 2]) - 32.0) / Om_s[:, 0]                   # (Omega_phi - Omega_b) / Omega_R for prograde orbits
for lo, hi, e1, e2 in [(0, 500, -1.35, -1.27), (500, 1000, -1.30, -1.22)]:
    s = (Lz_s > lo) & (Lz_s < hi) & (E_s > e1) & (E_s < e2)
    print(f'{lo} < Lz < {hi}, {e1} < E < {e2}: (Omega_phi - Omega_b)/Omega_R = {np.median(ratio_s[s]):+.2f} '
          f'(16-84%: {np.percentile(ratio_s[s], 16):+.2f} to {np.percentile(ratio_s[s], 84):+.2f})')
i_in, i_out = np.argmax(A_mod_prof[(0, 500)]), np.argmin(A_mod_prof[(500, 1000)])
assert -1.4 < E_mid[i_in] < -1.2 and A_mod_prof[(0, 500)][i_in] > 0.1            # the inward band of Figure 14.4
assert -1.35 < E_mid[i_out] < -1.2 and A_mod_prof[(500, 1000)][i_out] < -0.1       # and the outward one next to it

**Figure 14.8.** The inward–outward asymmetry $A$ as a function of energy $E$ in three bins of $L_z$: the slowing-bar model with
$\Omega_b = 32$ km/s/kpc today (solid; the band is the sampling error of the mean of $a_\star$), evaluated at 8000 Gaia stars and their
mirror images, and the data (dotted, the right panel of Figure 14.4). The dashed line marks the energy ridge of Figure 14.3.

The model reproduces both bands of Figure 14.4 (Figure 14.8). For $0 < L_z < 500$ kpc km/s it puts an inward excess of $A \approx +0.18$
at $E \approx -1.31 \times 10^5$ (km/s)$^2$ (data: $+0.21$ at $-1.29$), and for $500 < L_z < 1000$ an outward excess at $E \approx -1.26$
(model $-0.38$, data $-0.18$); for retrograde stars both stay within $\approx 0.08$ of zero. The correlation coefficients between the
model and data profiles are $+0.51$ and $+0.63$ in the two prograde bins. The orbits in both bands have $(\Omega_\phi - \Omega_b)/\Omega_R$
between $-0.06$ and $+0.03$ (16th–84th percentiles, printed above), i.e. they are at **corotation** with today's bar, the same
resonance that made the inward arm of the chevron in Figure 14.6. The match is not perfect: the model overestimates the outward
band and predicts weaker features the data do not show, an outward excess just below the inward band ($A \approx -0.07$ at
$E \approx -1.39$ for $0 < L_z < 500$) and an inward one at $E \approx -1.49$ for $500 < L_z < 1000$. As for Figure 14.7, the halo DF, the
slowing history and the potential are simple guesses. Note also that the asymmetric band lies above the energy ridge of Figure 14.3,
not on it: the ridge is an excess of stars moving in both directions, and the asymmetry measures only their difference.

## 14.4 The bar disperses substructure — but not in the Jacobi integral

Chapter 12 used the integrals of motion $E$ and $L_z$ to find debris that had phase-mixed in space: the stars of one disrupted
satellite stay together in $(L_z, E)$. In a rotating bar neither is conserved. Dillamore & Sanders (2025) showed that the bar
spreads low-energy substructure by factors of 10–100 in $L_z$ and $E$ — along lines of slope $dE/dL_z = \Omega_b$, because the Jacobi
integral $E_J = E - \Omega_b L_z$ (Chapter 13) is conserved, and so a change $\Delta L_z$ of a star's angular momentum comes with a change $\Delta E = \Omega_b \Delta L_z$ of its energy. We test this with a
compact clump of 2000 stars on a prograde, eccentric inner-halo orbit, integrated for 6 time units (5.9 Gyr) with and without a
bar rotating steadily at 35 km/s/kpc.

In [ ]:
Ob = 35.0
pot_rot = agama.Potential(potential=pot_bar, rotation=[[0, bar_angle], [1, bar_angle - Ob]])     # steady clockwise bar
centre = np.array([-5., 0., 0.5, 60., 120., 20.])                                                # a clump at r = 5 kpc
clump = np.column_stack([centre[:3] + rng.normal(0, 0.05, (2000, 3)), centre[3:] + rng.normal(0, 3, (2000, 3))])
spread = {}
for name, pot in [('no bar', pot_axi), ('bar', pot_rot)]:
    w = np.array([traj[-1] for traj in agama.orbit(potential=pot, ic=clump, time=6.0, trajsize=1, verbose=False)[:, 1]], float)
    E_w = 0.5 * np.sum(w[:, 3:]**2, axis=1) + pot.potential(w[:, :3], t=6.0)
    Lz_w = -(w[:, 0] * w[:, 4] - w[:, 1] * w[:, 3])                    # prograde
    spread[name] = (E_w, Lz_w, E_w - Ob * Lz_w)                         # E_J with prograde L_z and Omega_b > 0
E_in = 0.5 * np.sum(clump[:, 3:]**2, axis=1) + pot_rot.potential(clump[:, :3], t=0)   # the bar at its angle today
Lz_in = -(clump[:, 0] * clump[:, 4] - clump[:, 1] * clump[:, 3])

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name, colr in [('no bar', 'C0'), ('bar', 'C3')]:
    E_w, Lz_w, EJ_w = spread[name]
    ax[0].plot(Lz_w, E_w / 1e5, '.', ms=1.5, color=colr, label=f'after 5.9 Gyr, {name}')
    ax[1].hist(EJ_w / 1e5, 40, histtype='step', lw=2, color=colr, label=name)
ax[0].plot(Lz_in, E_in / 1e5, 'k.', ms=1.5, label='at the start')
Lline = np.linspace(200, 1100, 2)
ax[0].plot(Lline, (E_in.mean() + Ob * (Lline - Lz_in.mean())) / 1e5, 'k--', lw=1, label='slope $\\Omega_b$')
ax[0].set_xlabel('$L_z$ [kpc km/s]'); ax[0].set_ylabel('$E$ [$10^5$ (km/s)$^2$]'); ax[0].legend(fontsize=8, markerscale=6)
ax[1].set_xlabel('$E_J = E - \\Omega_b L_z$ [$10^5$ (km/s)$^2$]'); ax[1].set_ylabel('stars'); ax[1].legend()
plt.tight_layout(); plt.show()
sd = {k: [np.std(q) for q in v] for k, v in spread.items()}
print(f'spread after 5.9 Gyr (std): L_z {sd["no bar"][1]:.0f} -> {sd["bar"][1]:.0f} kpc km/s; '
      f'E {sd["no bar"][0]:.0f} -> {sd["bar"][0]:.0f}; E_J {sd["no bar"][2]:.0f} -> {sd["bar"][2]:.0f} (km/s)^2')
slope = np.polyfit(spread['bar'][1], spread['bar'][0], 1)[0]
print(f'slope dE/dL_z of the dispersed clump: {slope:.1f} km/s/kpc')
assert sd['bar'][1] > 5 * sd['no bar'][1] and sd['bar'][2] < 1.5 * sd['no bar'][2] and abs(slope - Ob) < 3

**Figure 14.9.** A clump of stars on a prograde inner-halo orbit. Left: $(L_z, E)$ at the start (black) and after 5.9 Gyr without (blue) and
with (red) a bar rotating at 35 km/s/kpc; the dashed line has slope $\Omega_b$. Right: distribution of the Jacobi integral after 5.9 Gyr.

Without the bar the clump stays compact in $(L_z, E)$ — only phase mixing in the slightly flattened potential spreads it a little. With
the bar its $L_z$ spreads seven times more, along a line of slope $\approx \Omega_b$ (Figure 14.9, left), while the spread of its Jacobi
integral stays as small as without the bar (right; the two histograms are offset because $E_J$ is measured in two different
potentials). For the inner halo — orbits that reach the bar — searches for substructure should use
$E_J$ (and chemistry) instead of $E$ and $L_z$; and the direction in which old substructure is smeared records the bar's pattern
speed, and its history (Dillamore & Sanders 2025).


## 14.5 The bar spins up the inner halo

So far the halo has been a record of the bar. But the bar also exerts a torque on halo stars and, over billions of years, gives
them angular momentum: an inner halo that started with no net rotation should end up rotating with the bar. We test this with a
forward experiment, from a non-rotating halo, for a bar of constant pattern speed and for a bar that slows down.

**The halo.** We take massless test particles from the non-rotating halo component of Dillamore et al. (2024, their §3.1). Its density is
$\rho \propto (r/r_s)^{-2.3}(1 + r/r_s)^{-2.3}$ with $r_s = 25$ kpc (an AGAMA `Spheroid` with `gamma=2.3`, `beta=4.6`, `alpha=1`: inner slope
2.3, outer slope 4.6, and a transition of unit sharpness), and its velocities are **isotropic** — equal dispersions in all directions —
described by a distribution function of the energy alone, $f(E)$, obtained from the density by Eddington inversion (Chapter 8; AGAMA's
`QuasiSpherical` DF with velocity anisotropy `beta0=0`). Because $f$ depends only on $E$, a star and its time-reverse $(\mathbf x, -\mathbf v)$
are equally likely, so the halo does not rotate; we sample it in such pairs, which makes its total $L_z$ exactly zero. AGAMA inverts
for $f(E)$ in a spherical version of the potential, so in the flattened Milky Way the halo is close to, but not exactly in, equilibrium;
a run without the bar is the control. Stars inside 0.5 kpc are left out: their orbital periods are $\sim 1$ Myr and they would
dominate the integration time. (Dillamore et al. add a rotating halo component; we leave it out, to start with no rotation at all.)

**A bar that grows without changing the mass.** The stellar potential of the model (a `CylSpline`, Chapter 13) is a sum of
azimuthal harmonics,

$$ \Phi(R, z, \phi) = \Phi_0(R, z) + \sum_{m \ge 1} \left[ \Phi_m^c(R, z) \cos m\phi + \Phi_m^s(R, z) \sin m\phi \right] , $$

where $\phi$ is the azimuth, $m$ the azimuthal wavenumber, and $\Phi_m^c$, $\Phi_m^s$ the amplitudes of the cosine and sine terms. The
$m = 0$ term is the azimuthal average, the axisymmetric $\Phi_{\rm axi}$; it contains all the mass. Every $m > 0$ term averages to zero around
every ring, and so does its density: the $m > 0$ part, $\Phi_{m>0}$, only moves mass from some azimuths to others — towards the bar's long
axis, away from the perpendicular direction — and adds none (by Gauss's theorem, it changes the mass inside no sphere). The potential

$$ \Phi(\mathbf x, t) = \Phi_{\rm axi}(\mathbf x) + s(t)\, \Phi_{m>0}\!\left(\mathsf R_{\phi_b(t)}\,\mathbf x / S(t)\right) $$

therefore has the same mass at every moment: as the bar grows, the disk and the bulge give up exactly the mass the bar gains. Here
$s(t)$ is the bar's strength, rising from 0 to 1; $\mathsf R_{\phi_b}$ rotates the bar to its angle $\phi_b(t) = -\int_0^t \Omega_b\,dt'$
(clockwise); and $S(t) = 39/\Omega_b(t)$ (with $\Omega_b$ in km/s/kpc) stretches it, so that its length scales as $1/\Omega_b$ — 39 km/s/kpc is the
pattern speed of the Portail et al. (2017) model behind our bar. This is the set-up of Dillamore et al. (2024, §3.1 and Appendix C),
who keep "the axisymmetric components constant so that the total mass is conserved". The strength grows as in Dehnen (2000),

$$ s(t) = \tfrac{3}{16}\xi^5 - \tfrac58 \xi^3 + \tfrac{15}{16}\xi + \tfrac12, \qquad \xi = 2t/t_1 - 1 \quad (t < t_1), \qquad s = 1 \quad (t \ge t_1), $$

with $t_1 = 2$ time units (1.96 Gyr), and the slowing bar follows their pattern speed,

$$ \Omega_b(t) = \begin{cases} \Omega_i & t < t_1 \\ \Omega_i \left[1 + \tfrac12 \eta\, \Omega_i (t - t_1)^2/(t_2 - t_1)\right]^{-1} & t_1 \le t < t_2 \\
   \Omega_2 \left[1 + \eta\, \Omega_2 (t - t_2)\right]^{-1} & t \ge t_2 \end{cases}, \qquad \Omega_2 = \Omega_i \left[1 + \tfrac12 \eta\, \Omega_i (t_2 - t_1)\right]^{-1} , $$

where $\Omega_i = 80$ km/s/kpc is the initial pattern speed, $\eta = 0.003$ the dimensionless deceleration rate of §14.3
($\dot\Omega_b = -\eta\,\Omega_b^2$ after $t_2$), $t_2 = 3$ time units the end of a smooth onset of the deceleration, and $\Omega_2$ the pattern
speed at $t_2$. Run for 7 time units (6.8 Gyr), the bar slows from 80 to 38.5 km/s/kpc. The comparison bar keeps $\Omega_b = 80$ km/s/kpc.

In AGAMA, $\Phi_{m>0}$ is made by exporting the `CylSpline` coefficients to a text file, setting the $m = 0$ block to zero and loading the
file back; the growth, stretching and rotation are the `scale` and `rotation` modifiers of §14.3.

In [ ]:
from scipy.integrate import cumulative_trapezoid


def m_positive_part(pot, fname='bar_m_gt_0.ini'):
    """The non-axisymmetric (m > 0) part of a CylSpline potential: export its coefficients, zero the m = 0 block, reload."""
    pot.export(fname)
    out, m = [], None
    for line in open(fname).read().splitlines():
        if line.endswith('#m'):                         # each block of coefficients starts with a line "<m>  #m"
            m = int(line.split()[0])
        elif m == 0 and line[:1].isdigit():             # a data row of the m = 0 block: keep R (1st column), zero the rest
            cols = line.split('\t')
            line = '\t'.join([cols[0]] + ['0'] * (len(cols) - 1))
        out.append(line)
    open(fname, 'w').write('\n'.join(out) + '\n')
    return agama.Potential(file=fname)


pot_m = m_positive_part(pot_bar[1])                     # pot_bar = [spheroids, CylSpline with mmax = 8]

T_run, t1, t2 = 7.0, 2.0, 3.0                           # run length, end of growth, end of the deceleration ramp [time units]


def omega_D24(t, Om_i=80.0, eta=0.003):
    """Pattern speed of Dillamore et al. (2024, Appendix C): constant Om_i while the bar grows (t < t1), a gradual onset of
    the deceleration for t1 < t < t2, then dOmega/dt = -eta Omega^2."""
    Om2 = Om_i / (1 + 0.5 * eta * Om_i * (t2 - t1))
    return np.where(t < t1, Om_i, np.where(t < t2, Om_i / (1 + 0.5 * eta * Om_i * (t - t1)**2 / (t2 - t1)),
                                           Om2 / (1 + eta * Om2 * (t - t2))))


def growing_bar(omega):
    """Phi_axi + s(t) Phi_{m>0}, rotating at omega(t) [clockwise] and stretched by S(t) = 39 / omega(t)."""
    tt = np.linspace(0, T_run + 0.5, 4001)
    Om = omega(tt)
    angle = -cumulative_trapezoid(Om, tt, initial=0)                       # phi_b(t) = -int Omega_b dt (clockwise)
    S = 39.0 / Om                                                          # Portail et al. (2017) bar has 39 km/s/kpc
    xi = np.clip(2 * tt / t1 - 1, -1, 1)
    s = 3 / 16 * xi**5 - 5 / 8 * xi**3 + 15 / 16 * xi + 0.5             # Dehnen (2000) growth, 0 -> 1 over [0, t1]
    return agama.Potential(pot_axi, agama.Potential(potential=pot_m, scale=np.column_stack([tt, s * S, S]),
                                                    rotation=np.column_stack([tt, angle])))


runs_spin = {'no bar': pot_axi,
             'constant, $\\Omega_b$ = 80': growing_bar(lambda t: np.full_like(t, 80.0)),
             f'slowing, 80 $\\to$ {omega_D24(T_run):.1f}': growing_bar(omega_D24)}

# tests: Phi_axi + Phi_{m>0} reproduces the barred model; Phi_{m>0} has zero mean around every ring (adds no mass)
p = np.random.default_rng(0).uniform(-8, 8, (2000, 3)) * [1, 1, 0.3]
err = np.max(np.abs((pot_axi.potential(p) + pot_m.potential(p)) / pot_bar.potential(p) - 1))
phi = np.linspace(0, 2 * np.pi, 361)[:-1]
ring_mean = max(abs(pot_m.potential(np.column_stack([Rr * np.cos(phi), Rr * np.sin(phi), 0 * phi])).mean()) for Rr in (1, 2, 4, 8))
print(f'Phi_axi + Phi_m>0 vs barred model: max relative difference {err:.1e}; largest ring average of Phi_m>0: {ring_mean:.1e} (km/s)^2')
print(f'pattern speed of the slowing bar: {omega_D24(np.array([0, t1, t2, T_run]))} km/s/kpc at t = 0, t1, t2, T_run')
assert err < 2e-4 and ring_mean < 1e-6
assert all(abs(pot.potential(p[:5], t=0) / pot_axi.potential(p[:5]) - 1).max() < 1e-12 for pot in runs_spin.values())

In [ ]:
agama.setRandomSeed(14)
halo_D24 = agama.DistributionFunction(type='QuasiSpherical', beta0=0, potential=pot_axi,
                                      density=agama.Density(type='Spheroid', densityNorm=1, gamma=2.3, beta=4.6, alpha=1,
                                                            scaleRadius=25.0))
xv_s, _ = agama.GalaxyModel(pot_axi, halo_D24).sample(10000)
xv_s = xv_s[np.linalg.norm(xv_s[:, :3], axis=1) > 0.5][:5000]          # skip the nuclear cusp: ~1 Myr orbits, slow to integrate
xv_s = np.vstack([xv_s, xv_s * [1, 1, 1, -1, -1, -1]])                  # each star and its time-reverse: total L_z = 0 exactly
Lz_of = lambda xv: -(xv[..., 0] * xv[..., 4] - xv[..., 1] * xv[..., 3])  # prograde L_z
print(f'{len(xv_s)} halo stars, median r = {np.median(np.linalg.norm(xv_s[:, :3], axis=1)):.1f} kpc, total L_z = {Lz_of(xv_s).sum():.1e}')

t_out = np.linspace(0, T_run, 71)                                        # every 0.1 time units
spin = {}
for lab, pot in runs_spin.items():
    t0 = time.time()
    out = agama.orbit(potential=pot, ic=xv_s, time=T_run, trajsize=len(t_out), verbose=False)
    spin[lab] = np.stack(out[:, 1]).astype(float)                       # (stars, times, 6)
    dLz = Lz_of(spin[lab][:, -1]) - Lz_of(xv_s)
    print(f'{lab:22s}: {time.time() - t0:3.0f} s; mean change of L_z {dLz.mean():+6.1f} kpc km/s per star')

In [ ]:
Rg = np.linspace(0.5, 20, 400)
Om_circ = np.sqrt(-Rg * pot_axi.force(np.column_stack([Rg, 0 * Rg, 0 * Rg]))[:, 0]) / Rg      # v_c(R) / R
R_CR = lambda Om: np.interp(-Om, -Om_circ, Rg)                                           # corotation radius
late = t_out >= T_run - 1.0                                                              # the last Gyr: 11 snapshots
R_edges = np.arange(0, 30.1, 2.0)
R_mid = 0.5 * (R_edges[1:] + R_edges[:-1])
fig, ax = plt.subplots(figsize=(9, 5))
vphi_prof = {}
for (lab, tr), colr in zip(spin.items(), ['0.5', 'C3', 'C0']):
    R_t = np.hypot(tr[:, late, 0], tr[:, late, 1])
    vphi = Lz_of(tr[:, late]) / R_t                                                      # v_phi = L_z / R, prograde > 0
    i = np.digitize(R_t, R_edges)
    vphi_prof[lab] = np.array([vphi[i == k].mean() for k in range(1, len(R_edges))])
    err = np.array([vphi[i == k].std() / np.sqrt((i == k).sum() / late.sum()) for k in range(1, len(R_edges))])
    ax.errorbar(R_mid, vphi_prof[lab], err, color=colr, marker='o', ms=4, capsize=2, label=lab)
for Om, ls in ((80.0, '--'), (omega_D24(T_run), ':')):
    ax.axvline(R_CR(Om), color='k', ls=ls, lw=1, label=f'corotation for $\\Omega_b$ = {Om:.1f}: {R_CR(Om):.1f} kpc')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('$R$ [kpc]'); ax.set_ylabel('mean $v_\\phi$ [km/s]'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
for lab, prof in vphi_prof.items():
    print(f'{lab:22s}: mean v_phi at R < 4 kpc {np.mean(prof[R_mid < 4]):+5.1f}, at 4-10 kpc {np.mean(prof[(R_mid > 4) & (R_mid < 10)]):+5.1f}, '
          f'at 10-30 kpc {np.mean(prof[R_mid > 10]):+5.1f} km/s')
labs = list(spin)
assert np.mean(vphi_prof[labs[2]][R_mid < 10]) > np.mean(vphi_prof[labs[1]][R_mid < 10]) + 5     # the slowing bar spins up more

**Figure 14.10.** Mean rotation velocity $v_\phi$ (positive in the direction of Galactic rotation) of the halo stars in bins of cylindrical
radius $R$, averaged over the last Gyr of the experiment (11 snapshots), for no bar (grey), a bar rotating at a constant 80 km/s/kpc (red) and a bar
slowing from 80 to 38.5 km/s/kpc (blue). Error bars count each star once. The vertical lines mark the corotation radius $R_{\rm CR}$, where the
circular angular velocity $v_c/R$ equals $\Omega_b$, for 80 (dashed) and 38.5 km/s/kpc (dotted).

Both bars spin up the halo, but only near the centre, and by very different amounts (Figure 14.10). The constant bar makes the halo
inside 4 kpc rotate at $\langle v_\phi \rangle \approx 4$ km/s and leaves it untouched further out. The slowing bar makes it rotate at
$\approx 15$ km/s inside 4 kpc and $\approx 11$ km/s at 4–10 kpc; beyond 10 kpc the rotation stays within $\approx 2$ km/s of zero. Averaged over
all stars, $L_z$ grows by 3.3 kpc km/s per star with the constant bar and by 40.8 with the slowing one, twelve times more. A bar of fixed
pattern speed exchanges angular momentum only with orbits near its resonances — which stay at fixed places, mostly inside its
corotation radius of 2.5 kpc — and once those orbits are rearranged the exchange stops. A slowing bar moves its resonances outwards
(corotation from 2.5 to 5.9 kpc) and drags captured stars along (Chiba et al. 2021), so the spin-up spreads to about the final corotation
radius. Exercise 6 asks you to check which resonances carry the gain. One caveat: our stars are massless. In a real galaxy the
angular momentum the halo gains is taken from the bar — dynamical friction, which is what slows bars down; here the slowing is imposed.

In [ ]:
E_axi = lambda xv: (pot_axi.potential(xv[:, :3]) + 0.5 * (xv[:, 3:]**2).sum(1)) / 1e5   # E in Phi_axi [1e5 (km/s)^2]
rng_L, rng_E = (-2000, 2000), (-2.4, -1.0)
dens = lambda xv: gaussian_filter(np.histogram2d(Lz_of(xv), E_axi(xv), bins=50, range=[rng_L, rng_E])[0], 1)
H0 = dens(xv_s)
H1 = {lab: dens(spin[lab][:, -1]) for lab in labs[1:]}
dmax = max(np.abs(H - H0).max() for H in H1.values())
fig, ax = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
kw = dict(origin='lower', aspect='auto', extent=[*rng_L, *rng_E])
for row, lab in enumerate(labs[1:]):
    ax[row, 0].imshow(np.log10(H0.T + 0.1), cmap='Greys', vmin=-1, vmax=np.log10(H0.max()), **kw)
    ax[row, 1].imshow(np.log10(H1[lab].T + 0.1), cmap='Greys', vmin=-1, vmax=np.log10(H0.max()), **kw)
    im = ax[row, 2].imshow((H1[lab] - H0).T, cmap='RdBu_r', vmin=-dmax, vmax=dmax, **kw)
    plt.colorbar(im, ax=ax[row, 2], label='stars per cell, final $-$ initial')
    ax[row, 0].set_title('initial'); ax[row, 1].set_title(f'{lab}: after {T_run * GYR:.1f} Gyr'); ax[row, 2].set_title('difference')
    ax[row, 0].set_ylabel('$E$ in $\\Phi_{\\rm axi}$ [$10^5$ (km/s)$^2$]')
for a in ax[1]:
    a.set_xlabel('$L_z$ [kpc km/s]')
plt.tight_layout(); plt.show()
for lab in labs[1:]:
    D = H1[lab] - H0
    Lc, Ec = np.meshgrid(np.linspace(*rng_L, 51)[:-1] + 40, np.linspace(*rng_E, 51)[:-1] + 0.014, indexing='ij')
    i = np.unravel_index(np.argmax(D), D.shape)
    print(f'{lab:22s}: largest gain {D[i]:.1f} stars per cell at L_z = {Lc[i]:.0f}, E = {Ec[i]:.2f}; '
          f'largest loss {D.min():.1f} at L_z = {Lc[np.unravel_index(np.argmin(D), D.shape)]:.0f}, '
          f'E = {Ec[np.unravel_index(np.argmin(D), D.shape)]:.2f}')

**Figure 14.11.** The halo in the $(L_z, E)$ plane, with $E$ computed in $\Phi_{\rm axi}$ at both times. Top: the bar with a constant 80 km/s/kpc;
bottom: the bar slowing from 80 to 38.5 km/s/kpc. Left: at the start (log density); middle: after 6.8 Gyr; right: the difference (red: more stars
than at the start), in stars per cell of 80 kpc km/s $\times$ 0.028 $\times 10^5$ (km/s)$^2$, smoothed by one cell.

The constant bar changes the $(L_z, E)$ distribution little, and only at low energy around $L_z \approx 0$ (Figure 14.11, top; at most
5 stars per cell). The slowing bar empties a strip of non-rotating stars at $E \approx -2.0$ to $-1.6 \times 10^5$ (km/s)$^2$ and moves them
into a prograde clump at $E \approx -1.38 \times 10^5$ (km/s)$^2$, $L_z \approx 300$–1200 kpc km/s (bottom), with a weaker band near
$E \approx -1.15$. The stars gain energy and angular momentum together: from the largest loss to the largest gain the slope is
$\Delta E / \Delta L_z \approx 70$ km/s/kpc, between the initial and final pattern speeds, as expected from $\Delta E = \Omega_b \Delta L_z$ (§14.4) for a
bar that has slowed while the stars were captured. The energy of the clump is close to that of the ridge in the Gaia data (Figure 14.3,
$-1.43 \times 10^5$); with an untuned halo, slowing history and final pattern speed this is suggestive, not a result (Exercise 7).

## Summary

- The local stellar halo is asymmetric between inward and outward motion: non-rotating stars have an excess moving
  inwards (8 standard deviations), retrograde stars none; the asymmetry maps in $(r, |v_r|)$ show chevron arms, and the
  $(L_z, E)$ plane a ridge at constant energy, with the inward excess concentrated in a narrow band of energy just above it.
- Phase-mixed merger debris makes chevrons, but symmetric ones, spread in energy.
- A slowing bar, modelled by backward integration, traps halo stars at its resonances and makes asymmetric chevrons along
  the resonance lines. Matching the data favours a present-day $\Omega_b \approx 32$–34 km/s/kpc.
  Evaluated at the Gaia stars themselves, the same model reproduces the inward and outward bands in $(L_z, E)$, which lie at corotation.
- The bar exchanges energy and angular momentum with halo stars along $\Delta E = \Omega_b \Delta L_z$; substructure disperses in $(L_z, E)$
  but not in the Jacobi integral.
- A bar grown in a non-rotating halo spins up its inner part. A bar of constant pattern speed (80 km/s/kpc) gives
  $\langle v_\phi \rangle \approx 4$ km/s inside 4 kpc; a bar slowing from 80 to 38.5 km/s/kpc gives 11–15 km/s out to $\approx 8$ kpc, twelve times more
  angular momentum, and moves non-rotating stars into a prograde clump in $(L_z, E)$.

| task | AGAMA |
|---|---|
| a bar whose speed changes | `agama.Potential(potential=pot, rotation=np.column_stack([t, angle(t)]))` |
| a bar that grows or stretches | `agama.Potential(potential=pot, scale=np.column_stack([t, amplitude(t), length_factor(t)]))` |
| a halo DF of actions | `agama.DistributionFunction(type='DoublePowerLaw', ...)`, evaluated as `df(agama.ActionFinder(pot)(xv))` |
| backward integration | `agama.orbit(potential=pot, ic=ic, time=-T, timestart=0, trajsize=1, verbose=False)` |
| frequencies for resonance lines | `agama.ActionFinder(pot)(xv, frequencies=True)` |
| a halo DF of the energy only | `agama.DistributionFunction(type='QuasiSpherical', density=dens, potential=pot, beta0=0)`, sampled by `agama.GalaxyModel(pot, df).sample(N)` |
| the non-axisymmetric part of a `CylSpline` | `pot.export(file)`, zero the `0 #m` block, `agama.Potential(file=file)` |

## Exercises

1. **Significance.** Repeat Figure 14.2 for a sample in which $v_r$ of each star is replaced by $\pm v_r$ with a random sign. What
   level of $|A|$ does noise alone produce? Which features of Figure 14.2 survive?
2. **A steady bar.** Recompute Figure 14.7 with the constant pattern speed of Chapter 13 (`rotation=[[0, bar_angle], [1, bar_angle - 35]]`
   and no stretching). Are the chevrons weaker? Chiba et al. (2021) and Dillamore et al. (2024) explain why.
3. **The other bins.** Make the model maps for $500 < L_z < 1000$ and for retrograde $L_z$ and compare them with Figure 14.2.
4. **The energy ridge.** Compute the energy of the corotation resonance for $\Omega_b = 32$, 35 and 38 km/s/kpc for orbits with
   $0 < L_z < 500$ (use the `ActionFinder` frequencies on a grid of $E$). Which pattern speed puts corotation at the ridge of Figure 14.3?
5. **Chemistry.** The file contains the XP metallicity `feh` of Zhang et al. (2023). Do the inward-moving stars of Figure 14.1 have
   the metallicity distribution of Gaia-Sausage/Enceladus (Chapter 12), or of the disk?
6. **Which resonances spin up the halo?** For the stars of §14.5, compute $(\Omega_\phi - \Omega_b)/\Omega_R$ at the end of the slowing-bar run
   (`ActionFinder` frequencies in $\Phi_{\rm axi}$, $\Omega_b = 38.5$ km/s/kpc) and plot the change of $L_z$ against it. Which resonances carry the gain?
7. **A slower bar today.** Change $\eta$ so that the bar ends at 32–35 km/s/kpc (the values favoured in §14.3). Does the prograde clump of
   Figure 14.11 move in energy, and how does it compare with the ridge of Figure 14.3?

## References

- V. Belokurov et al. (2023), "Energy wrinkles and phase-space folds of the last major merger", MNRAS 518, 6200:
  https://ui.adsabs.harvard.edu/abs/2023MNRAS.518.6200B
- R. Chiba, J. K. S. Friske & R. Schönrich (2021), "Resonance sweeping by a decelerating Galactic bar", MNRAS 500, 4710:
  https://ui.adsabs.harvard.edu/abs/2021MNRAS.500.4710C
- J. P. Clarke & O. Gerhard (2022), MNRAS 512, 2171: https://ui.adsabs.harvard.edu/abs/2022MNRAS.512.2171C
- E. Y. Davies, A. M. Dillamore & V. Belokurov (2023), "Accelerated phase-mixing in the stellar halo due to a rotating bar",
  MNRAS 521, L24: https://ui.adsabs.harvard.edu/abs/2023MNRAS.521L..24D
- W. Dehnen (2000), AJ 119, 800: https://ui.adsabs.harvard.edu/abs/2000AJ....119..800D
- A. M. Dillamore et al. (2023), "Stellar halo substructure generated by bar resonances", MNRAS 524, 3596:
  https://ui.adsabs.harvard.edu/abs/2023MNRAS.524.3596D
- A. M. Dillamore, V. Belokurov & N. W. Evans (2024), "Radial halo substructure in harmony with the Galactic bar", MNRAS 532, 4389:
  https://ui.adsabs.harvard.edu/abs/2024MNRAS.532.4389D
- A. M. Dillamore et al. (2025), "Dynamical streams in the local stellar halo", MNRAS 541, 214:
  https://ui.adsabs.harvard.edu/abs/2025MNRAS.541..214D
- A. M. Dillamore & J. L. Sanders (2025), "Bar-driven dispersal of Galactic substructure", MNRAS 542, 1331:
  https://ui.adsabs.harvard.edu/abs/2025MNRAS.542.1331D
- G. H. Hunter et al. (2024), A&A 692, A216: https://ui.adsabs.harvard.edu/abs/2024A&A...692A.216H
- M. Portail et al. (2017), "Dynamical modelling of the galactic bulge and bar", MNRAS 465, 1621:
  https://ui.adsabs.harvard.edu/abs/2017MNRAS.465.1621P
- M. C. Sormani et al. (2022), MNRAS 514, L1: https://ui.adsabs.harvard.edu/abs/2022MNRAS.514L...1S
- X. Zhang, G. M. Green & H.-W. Rix (2023), "Parameters of 220 million stars from Gaia BP/RP spectra", MNRAS 524, 1855:
  https://ui.adsabs.harvard.edu/abs/2023MNRAS.524.1855Z
- Data: V. Belokurov (2026), "Galactic Dynamics Tutorial data", Zenodo, https://doi.org/10.5281/zenodo.23646482 — please also cite
  the sources listed in its README (Gaia DR3, Katz et al. 2023, Zhang et al. 2023).